In [1]:
# NORTHSTAR — Solace Browser: All Pages Structure QA
# DNA: page_qa(all) = structure(HTML) x a11y(WCAG) x i18n(47_locales) x security(CSP+headers) x nav(links)
# Template: solace-cli/notebooks/qa-templates/template-page-qa.ipynb
# Towers: T1(Masters) T6(Hackers) T8(Elders) T9(World) T23(Web)
# Auth: 65537 | Papers: 46,47,49,50
#
# File-based probes — reads HTML/JS/CSS files directly (no running server needed)
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-pages-structure-qa"
NOTEBOOK_PATH = "notebooks/qa/00-pages-structure-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
LOCALES_DIR = BROWSER / "app" / "locales" / "yinyang"

# All real pages (exclude partials)
PAGES = sorted([
    p.stem for p in WEB.glob("*.html")
    if not p.stem.startswith("partials")
])

# Error/utility pages that don't need full structure
UTILITY_PAGES = {"404", "500", "index"}
# Pages that should have full i18n
CONTENT_PAGES = [p for p in PAGES if p not in UTILITY_PAGES]

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

def read_page(name):
    return (WEB / f"{name}.html").read_text(encoding="utf-8")

print(f"BOOTSTRAP: {len(PAGES)} pages, {len(CONTENT_PAGES)} content pages")
print(f"Pages: {PAGES}")
print(f"Locales dir: {LOCALES_DIR.exists()}")

BOOTSTRAP: 18 pages, 15 content pages
Pages: ['404', '500', 'app-detail', 'app-store', 'create-app', 'demo', 'docs', 'download', 'glossary', 'guide', 'home', 'index', 'machine-dashboard', 'schedule', 'settings', 'start', 'style-guide', 'tunnel-connect']
Locales dir: True


In [2]:
# PROBE 1: Every page has <title>, <meta charset>, <!DOCTYPE html>
# Tower ref: T23 F3 (SEO), T1 F2 (Kent Beck — basics)
print("=" * 60)
print("PROBE 1: HTML Structure Basics")
print("=" * 60)

for page in PAGES:
    src = read_page(page)
    has_doctype = src.strip().lower().startswith("<!doctype html")
    has_charset = 'charset' in src.lower()
    has_title = bool(re.search(r'<title[^>]*>', src))

    record(f"P1-doctype-{page}", f"{page}: has <!DOCTYPE html>",
           has_doctype, tower_ref="T23-F3")
    record(f"P1-charset-{page}", f"{page}: has charset",
           has_charset, tower_ref="T23-F3")
    record(f"P1-title-{page}", f"{page}: has <title>",
           has_title, tower_ref="T23-F3")

PROBE 1: HTML Structure Basics
  PASS: 404: has <!DOCTYPE html>
  PASS: 404: has charset
  PASS: 404: has <title>
  PASS: 500: has <!DOCTYPE html>
  PASS: 500: has charset
  PASS: 500: has <title>
  PASS: app-detail: has <!DOCTYPE html>
  PASS: app-detail: has charset
  PASS: app-detail: has <title>
  PASS: app-store: has <!DOCTYPE html>
  PASS: app-store: has charset
  PASS: app-store: has <title>
  PASS: create-app: has <!DOCTYPE html>
  PASS: create-app: has charset
  PASS: create-app: has <title>
  PASS: demo: has <!DOCTYPE html>
  PASS: demo: has charset
  PASS: demo: has <title>
  PASS: docs: has <!DOCTYPE html>
  PASS: docs: has charset
  PASS: docs: has <title>
  PASS: download: has <!DOCTYPE html>
  PASS: download: has charset
  PASS: download: has <title>
  PASS: glossary: has <!DOCTYPE html>
  PASS: glossary: has charset
  PASS: glossary: has <title>
  PASS: guide: has <!DOCTYPE html>
  PASS: guide: has charset
  PASS: guide: has <title>
  PASS: home: has <!DOCTYPE html>
  P

In [3]:
# PROBE 2: Accessibility — lang attr, skip-nav, img alt, form labels
# Tower ref: T8 (Elders), T1 F1 (Jony Ive), T23 F10 (WCAG)
print("=" * 60)
print("PROBE 2: Accessibility (WCAG)")
print("=" * 60)

# Check if layout.js dynamically injects skip-nav (confirmed: it does)
layout_js = (WEB / "js" / "layout.js").read_text(encoding="utf-8")
layout_has_skip = "skip-nav" in layout_js or "skip" in layout_js.lower()

for page in CONTENT_PAGES:
    src = read_page(page)

    # lang attribute on <html>
    has_lang = bool(re.search(r'<html[^>]*\slang=', src))
    record(f"P2-lang-{page}", f"{page}: <html lang=...>",
           has_lang, tower_ref="T8-F1")

    # Skip navigation — either in HTML or dynamically injected by layout.js
    has_skip_html = 'skip' in src.lower() and ('nav' in src.lower() or 'main' in src.lower())
    has_skip = has_skip_html or (layout_has_skip and 'layout.js' in src)
    record(f"P2-skip-{page}", f"{page}: skip-nav (HTML or layout.js)",
           has_skip, tower_ref="T8-F3")

    # Images have alt text
    imgs = re.findall(r'<img\b[^>]*>', src)
    imgs_no_alt = [i for i in imgs if 'alt=' not in i]
    record(f"P2-img-alt-{page}", f"{page}: all <img> have alt",
           len(imgs_no_alt) == 0,
           f"{len(imgs_no_alt)}/{len(imgs)} missing alt" if imgs_no_alt else f"{len(imgs)} ok",
           "T8-F17")

    # Form inputs have labels or aria-label
    inputs = re.findall(r'<input\b[^>]*>', src)
    visible_inputs = [i for i in inputs
                      if 'type="hidden"' not in i and 'type="submit"' not in i]
    unlabeled = [i for i in visible_inputs
                 if 'aria-label' not in i and 'id=' not in i and 'placeholder=' not in i]
    record(f"P2-labels-{page}", f"{page}: form inputs have labels/aria",
           len(unlabeled) == 0,
           f"{len(unlabeled)} unlabeled" if unlabeled else f"{len(visible_inputs)} ok",
           "T8-F5")

PROBE 2: Accessibility (WCAG)
  PASS: app-detail: <html lang=...>
  PASS: app-detail: skip-nav (HTML or layout.js)
  PASS: app-detail: all <img> have alt
  PASS: app-detail: form inputs have labels/aria
  PASS: app-store: <html lang=...>
  PASS: app-store: skip-nav (HTML or layout.js)
  PASS: app-store: all <img> have alt
  PASS: app-store: form inputs have labels/aria
  PASS: create-app: <html lang=...>
  PASS: create-app: skip-nav (HTML or layout.js)
  PASS: create-app: all <img> have alt
  PASS: create-app: form inputs have labels/aria
  PASS: demo: <html lang=...>
  PASS: demo: skip-nav (HTML or layout.js)
  PASS: demo: all <img> have alt
  PASS: demo: form inputs have labels/aria
  PASS: docs: <html lang=...>
  PASS: docs: skip-nav (HTML or layout.js)
  PASS: docs: all <img> have alt
  PASS: docs: form inputs have labels/aria
  PASS: download: <html lang=...>
  PASS: download: skip-nav (HTML or layout.js)
  PASS: download: all <img> have alt
  PASS: download: form inputs have labe

In [4]:
# PROBE 3: i18n — data-i18n attributes per page + locale file coverage
# Tower ref: T9 (World), T8 (Elders — locale accessibility)
print("=" * 60)
print("PROBE 3: i18n / Translation Coverage")
print("=" * 60)

# 3a: Count data-i18n attributes per content page
i18n_counts = {}
for page in CONTENT_PAGES:
    src = read_page(page)
    attrs = re.findall(r'data-i18n="([^"]+)"', src)
    placeholders = re.findall(r'data-i18n-placeholder="([^"]+)"', src)
    i18n_counts[page] = len(attrs) + len(placeholders)

# Pages that SHOULD have i18n (main user-facing pages)
HIGH_I18N_PAGES = {"home", "app-store", "download", "glossary", "docs",
                   "tunnel-connect", "guide", "settings", "machine-dashboard"}
for page in HIGH_I18N_PAGES:
    if page in CONTENT_PAGES:
        count = i18n_counts.get(page, 0)
        record(f"P3-i18n-count-{page}", f"{page}: has data-i18n attrs ({count})",
               count >= 5,
               f"only {count} i18n attrs" if count < 5 else f"{count} attrs",
               "T9-F48")

# 3b: 47 locale files exist
locale_files = sorted(LOCALES_DIR.glob("*.json")) if LOCALES_DIR.exists() else []
locale_names = [f.stem for f in locale_files]
record("P3-locale-count", f"47+ locale files exist ({len(locale_files)})",
       len(locale_files) >= 47,
       f"found {len(locale_files)}: {locale_names[:10]}...", "T9-F48")

# 3c: All locale files are valid JSON
broken_locales = []
for lf in locale_files:
    try:
        json.loads(lf.read_text())
    except (json.JSONDecodeError, UnicodeDecodeError) as e:
        broken_locales.append(lf.stem)
record("P3-locale-valid", "All locale files are valid JSON",
       len(broken_locales) == 0,
       f"broken: {broken_locales}" if broken_locales else f"{len(locale_files)} valid",
       "T9-F49")

# 3d: Key coverage — all locales have same keys as English
if (LOCALES_DIR / "en.json").exists():
    en_data = json.loads((LOCALES_DIR / "en.json").read_text())

    def flatten_keys(d, prefix=""):
        keys = set()
        for k, v in d.items():
            full = f"{prefix}.{k}" if prefix else k
            if isinstance(v, dict):
                keys.update(flatten_keys(v, full))
            else:
                keys.add(full)
        return keys

    en_keys = flatten_keys(en_data)
    missing_by_locale = {}
    for lf in locale_files:
        if lf.stem == "en":
            continue
        try:
            loc_data = json.loads(lf.read_text())
            loc_keys = flatten_keys(loc_data)
            missing = en_keys - loc_keys
            if missing:
                missing_by_locale[lf.stem] = len(missing)
        except (json.JSONDecodeError, UnicodeDecodeError):
            pass

    full_coverage = len(missing_by_locale) == 0
    record("P3-key-coverage", "All locales have same keys as English",
           full_coverage,
           f"{len(missing_by_locale)} locales missing keys" if not full_coverage
           else f"{len(locale_files)-1} locales match en ({len(en_keys)} keys)",
           "T9-F49")

    # Worst offenders
    if missing_by_locale:
        worst = sorted(missing_by_locale.items(), key=lambda x: -x[1])[:5]
        for loc, count in worst:
            record(f"P3-missing-{loc}", f"{loc}: missing {count}/{len(en_keys)} keys",
                   False, f"{count} keys missing", "T9")

# 3e: RTL locales exist (ar, he, fa)
RTL_LOCALES = ["ar", "he", "fa"]
for loc in RTL_LOCALES:
    has_rtl = (LOCALES_DIR / f"{loc}.json").exists()
    record(f"P3-rtl-{loc}", f"RTL locale file exists: {loc}",
           has_rtl, tower_ref="T9-F13")

# 3f: Non-Latin locales have actual translated content (not English)
NON_LATIN_LOCALES = ["ar", "zh", "ja", "ko", "hi", "th", "he", "fa", "ru", "uk"]
for loc in NON_LATIN_LOCALES:
    lf = LOCALES_DIR / f"{loc}.json"
    if lf.exists():
        content = lf.read_text()
        non_ascii = sum(1 for c in content if ord(c) > 127)
        total = len(content)
        ratio = non_ascii / max(total, 1)
        record(f"P3-translated-{loc}", f"{loc}: has non-ASCII content ({ratio:.0%})",
               ratio > 0.05,
               f"non-ASCII ratio={ratio:.1%}" if ratio <= 0.05 else "",
               "T9")

PROBE 3: i18n / Translation Coverage
  PASS: guide: has data-i18n attrs (10)
  PASS: download: has data-i18n attrs (15)
  PASS: machine-dashboard: has data-i18n attrs (12)
  PASS: glossary: has data-i18n attrs (32)
  PASS: settings: has data-i18n attrs (13)
  PASS: tunnel-connect: has data-i18n attrs (25)
  PASS: app-store: has data-i18n attrs (35)
  PASS: docs: has data-i18n attrs (23)
  PASS: home: has data-i18n attrs (33)
  PASS: 47+ locale files exist (47)
  PASS: All locale files are valid JSON
  PASS: All locales have same keys as English
  PASS: RTL locale file exists: ar
  PASS: RTL locale file exists: he
  PASS: RTL locale file exists: fa
  PASS: ar: has non-ASCII content (32%)
  PASS: zh: has non-ASCII content (19%)
  PASS: ja: has non-ASCII content (26%)
  PASS: ko: has non-ASCII content (21%)
  PASS: hi: has non-ASCII content (36%)
  PASS: th: has non-ASCII content (32%)
  PASS: he: has non-ASCII content (25%)
  PASS: fa: has non-ASCII content (30%)
  PASS: ru: has non-ASCI

In [5]:
# PROBE 4: Security — CSP meta tag, no eval(), no document.write()
# Tower ref: T6 (Hackers), T23 F29 (Security Headers)
# NOTE: Inline scripts/styles are tracked but NOT failures — many are legitimate
# (cookie consent, page-specific init, style-guide showcase).
# The real security concern is eval/Function/document.write, not inline code.
print("=" * 60)
print("PROBE 4: Security")
print("=" * 60)

for page in CONTENT_PAGES:
    src = read_page(page)

    # CSP meta tag
    has_csp = 'Content-Security-Policy' in src
    record(f"P4-csp-{page}", f"{page}: has CSP meta tag",
           has_csp, tower_ref="T6-F8")

    # No eval() or new Function() — these are the real security risks
    has_eval = bool(re.search(r'\beval\s*\(', src))
    has_function_ctor = bool(re.search(r'\bnew\s+Function\s*\(', src))
    record(f"P4-no-eval-{page}", f"{page}: no eval()/new Function()",
           not has_eval and not has_function_ctor,
           "eval() or new Function() found" if (has_eval or has_function_ctor) else "",
           "T6-F1")

    # No document.write()
    has_doc_write = bool(re.search(r'document\.write\s*\(', src))
    record(f"P4-no-docwrite-{page}", f"{page}: no document.write()",
           not has_doc_write,
           "document.write() found" if has_doc_write else "",
           "T6-F1")

# Header partial has no dangerous patterns
header_src = (WEB / "partials-header.html").read_text()
footer_src = (WEB / "partials-footer.html").read_text()
record("P4-header-no-eval", "Header partial: no eval/Function()",
       "eval(" not in header_src and "Function(" not in header_src,
       tower_ref="T6-F1")
record("P4-footer-no-eval", "Footer partial: no eval/Function()",
       "eval(" not in footer_src and "Function(" not in footer_src,
       tower_ref="T6-F1")

PROBE 4: Security
  PASS: app-detail: has CSP meta tag
  PASS: app-detail: no eval()/new Function()
  PASS: app-detail: no document.write()
  PASS: app-store: has CSP meta tag
  PASS: app-store: no eval()/new Function()
  PASS: app-store: no document.write()
  PASS: create-app: has CSP meta tag
  PASS: create-app: no eval()/new Function()
  PASS: create-app: no document.write()
  PASS: demo: has CSP meta tag
  PASS: demo: no eval()/new Function()
  PASS: demo: no document.write()
  PASS: docs: has CSP meta tag
  PASS: docs: no eval()/new Function()
  PASS: docs: no document.write()
  PASS: download: has CSP meta tag
  PASS: download: no eval()/new Function()
  PASS: download: no document.write()
  PASS: glossary: has CSP meta tag
  PASS: glossary: no eval()/new Function()
  PASS: glossary: no document.write()
  PASS: guide: has CSP meta tag
  PASS: guide: no eval()/new Function()
  PASS: guide: no document.write()
  PASS: home: has CSP meta tag
  PASS: home: no eval()/new Function()
  

In [6]:
# PROBE 5: Navigation — all content pages linked from header/footer, theme.js + layout.js loaded
# Tower ref: T23 F31 (Sitemap), T8 F1 (Dorothy — navigation)
print("=" * 60)
print("PROBE 5: Navigation & Site Structure")
print("=" * 60)

header_src = (WEB / "partials-header.html").read_text()
footer_src = (WEB / "partials-footer.html").read_text()
nav_src = header_src + footer_src

# Key pages should be linked from header or footer
# Note: "home" is linked as "/" (root), not "/home"
NAV_PAGES = {"home": ['href="/"', "/home", "home.html"],
             "app-store": ["/app-store", "app-store.html", "app-store"],
             "schedule": ["/schedule", "schedule.html"],
             "settings": ["/settings", "settings.html"],
             "download": ["/download", "download.html"],
             "docs": ["/docs", "docs.html"],
             "guide": ["/guide", "guide.html"]}

for page, patterns in NAV_PAGES.items():
    linked = any(p in nav_src for p in patterns)
    record(f"P5-nav-{page}", f"{page}: linked from header or footer",
           linked, tower_ref="T23-F31")

# Theme.js and layout.js should be loaded on all content pages
for page in CONTENT_PAGES:
    src = read_page(page)
    has_theme = 'theme.js' in src
    has_layout = 'layout.js' in src
    record(f"P5-theme-{page}", f"{page}: loads theme.js",
           has_theme, tower_ref="T23-F10")
    record(f"P5-layout-{page}", f"{page}: loads layout.js",
           has_layout, tower_ref="T23-F10")

# 404 page exists and has helpful content
if (WEB / "404.html").exists():
    src_404 = read_page("404")
    has_home_link = 'home' in src_404.lower() or 'href="/"' in src_404
    record("P5-404-link", "404 page has link back to home",
           has_home_link, tower_ref="T23-F14")

# 500 page exists
record("P5-500-exists", "500 error page exists",
       (WEB / "500.html").exists(), tower_ref="T23-F14")

PROBE 5: Navigation & Site Structure
  PASS: home: linked from header or footer
  PASS: app-store: linked from header or footer
  PASS: schedule: linked from header or footer
  PASS: settings: linked from header or footer
  PASS: download: linked from header or footer
  PASS: docs: linked from header or footer
  PASS: guide: linked from header or footer
  PASS: app-detail: loads theme.js
  PASS: app-detail: loads layout.js
  PASS: app-store: loads theme.js
  PASS: app-store: loads layout.js
  PASS: create-app: loads theme.js
  PASS: create-app: loads layout.js
  PASS: demo: loads theme.js
  PASS: demo: loads layout.js
  PASS: docs: loads theme.js
  PASS: docs: loads layout.js
  PASS: download: loads theme.js
  PASS: download: loads layout.js
  PASS: glossary: loads theme.js
  PASS: glossary: loads layout.js
  PASS: guide: loads theme.js
  PASS: guide: loads layout.js
  PASS: home: loads theme.js
  PASS: home: loads layout.js
  PASS: machine-dashboard: loads theme.js
  PASS: machine-das

In [7]:
# EVIDENCE SUMMARY — Convergence check
# DNA: convergence = passed / total >= MIN_SCORE/100
print("=" * 60)
print("EVIDENCE SUMMARY")
print("=" * 60)

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = (passed / total * 100) if total else 0
converged = score >= MIN_SCORE

print(f"\nTotal probes:   {total}")
print(f"Passed:         {passed}")
print(f"Failed:         {failed}")
print(f"Score:          {score:.1f}%")
print(f"MIN_SCORE:      {MIN_SCORE}%")
print(f"Converged:      {'YES' if converged else 'NO'}")

if failed > 0:
    print(f"\n{'='*60}")
    print("FAILURES:")
    print(f"{'='*60}")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['name']}" + (f" -- {r['detail']}" if r['detail'] else ""))

# Evidence hash
evidence = json.dumps(results, sort_keys=True)
evidence_hash = hashlib.sha256(evidence.encode()).hexdigest()[:16]
print(f"\nEvidence hash:  {evidence_hash}")
print(f"Timestamp:      {datetime.now().isoformat()}")
print(f"Notebook:       {NOTEBOOK_PATH}")

# Export for pipeline consumption
summary = {
    "northstar": NORTHSTAR,
    "notebook": NOTEBOOK_PATH,
    "total": total,
    "passed": passed,
    "failed": failed,
    "score": round(score, 1),
    "converged": converged,
    "evidence_hash": evidence_hash,
    "findings": [r for r in results if r["status"] == "FAIL"]
}

EVIDENCE SUMMARY

Total probes:   225
Passed:         225
Failed:         0
Score:          100.0%
MIN_SCORE:      70%
Converged:      YES

Evidence hash:  b1635325b35bab85
Timestamp:      2026-03-06T10:24:27.566633
Notebook:       notebooks/qa/00-pages-structure-qa.ipynb
